In [ ]:
import pandas as pd

In [ ]:
cpc = pd.read_excel('CompanyPrimaryContact.xlsx')
iah = pd.read_excel('IndividualAffiliationHistory.xlsx')
ieh = pd.read_excel('IndividualEmploymentHistory.xlsx')
ipc = pd.read_excel('IndividualPrimaryContact.xlsx')

In [ ]:
iah

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

# Drop any unnecessary columns
iah = iah[["PID", "Organization", "Year Start", "Year End"]]

# Sort by organization and start year for easier processing
iah = iah.sort_values(by=["Organization", "Year Start"])


In [ ]:
from itertools import combinations

# Initialize graph
G = nx.Graph()

# Dictionary to store connections
connections = set()

# Group by organization and find connections
for org, group in iah.groupby("Organization"):
    employees = group.to_dict("records")
    
    for person1, person2 in combinations(employees, 2):
        pid1, start1, end1 = person1["PID"], person1["Year Start"], person1["Year End"]
        pid2, start2, end2 = person2["PID"], person2["Year Start"], person2["Year End"]

        # Check for overlap in work years
        if max(start1, start2) <= min(end1, end2):  
            key = tuple(sorted([pid1, pid2]))  # Ensure consistent key ordering
            
            if key not in connections:
                connections.add(key)

# Add nodes and edges to graph
for pid1, pid2 in connections:
    G.add_edge(pid1, pid2)

# Initialize Pyvis Network
net = Network(notebook=True, height="2200px", width="2800px", bgcolor="#222222", font_color="white")

# Apply physics settings to prevent node overlap
net.barnes_hut(gravity=-10000, central_gravity=0.3, spring_length=200, spring_strength=0.05, damping=0.4)

# Add nodes with size based on degree (number of connections)
for node in G.nodes():
    degree = G.degree(node)
    net.add_node(node, size=degree*5)  # Scaling size by degree

# Add edges
for edge in G.edges():
    net.add_edge(edge[0], edge[1])

# Save and display the network graph
net.show("connections.html")
